# Gradnja aplikacij za generiranje slik (OpenAI)

Modeli velikih jezikov (LLM) niso namenjeni le generiranju besedil. Prav tako lahko ustvarjate slike iz tekstovnih opisov. Slike kot modaliteta so uporabne v MedTech, arhitekturi, turizmu, razvoju iger, marketingu in drugih področjih. V tej lekciji delamo z današnjimi **GPT Image** modeli na platformi OpenAI.

## Cilji učenja

Do konca te lekcije boste znali:

- Pojasniti, kaj je generiranje slik in kje je uporabno.
- Razumeti družino modelov `gpt-image` in kako se razlikuje od starejših modelov DALL·E.
- Zgraditi aplikacijo za generiranje slik in urejati slike.

## Kaj je generiranje slik?

Modeli za generiranje slik ustvarjajo slike na podlagi tekstovnega poziva. Sodobni modeli, kot je `gpt-image`, se med učenjem naučijo razmerje med besedilom in slikami, nato pa postopoma spremenijo naključni šum v sliko, ki ustreza vašemu pozivu.

Dve dobro znani družini modelov za slike sta:

- **`gpt-image` (OpenAI)** — trenutno generacija, uporabljena v tej lekciji. Podpira generiranje slik iz besedila in urejanje slik (popravilo s masko).
- **Midjourney** — priljubljen model tretjih oseb s svojo storitvijo in delovnim procesom preko Discorda.

> Starejši OpenAI modeli za slike — **DALL·E 2** in **DALL·E 3** — so zastareli. DALL·E 3 ni več na voljo za nove uvedbe, funkcije, kot je `create_variation`, so bile prisotne samo v DALL·E 2. Za nove aplikacije uporabite modele `gpt-image`.

> **Pomembno:** modeli `gpt-image` vrnejo generirano sliko kot **base64** (`b64_json`), ne kot URL. Vaša koda pretvori base64 niz v bajte in jih shrani — ni URL-ja slike za prenos.


## Izgradnja vaše prve aplikacije za generiranje slik

Kaj torej potrebujete za izdelavo aplikacije za generiranje slik? Potrebujete naslednje knjižnice:

- **python-dotenv**, močno priporočamo uporabo te knjižnice za shranjevanje vaših skrivnosti v datoteko *.env* ločeno od kode.
- **openai**, ta knjižnica se uporablja za interakcijo z OpenAI API-jem.
- **pillow**, za delo s slikami v Pythonu.


1. Ustvarite datoteko *.env* z naslednjo vsebino:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Zberite zgornje knjižnice v datoteko z imenom *requirements.txt* tako:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Nato ustvarite virtualno okolje in namestite knjižnice:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Za Windows uporabite naslednje ukaze za ustvarjanje in aktivacijo vašega virtualnega okolja:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Dodajte naslednjo kodo v datoteko z imenom *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Pojasnimo to kodo:

- Najprej uvozimo knjižnice, ki jih potrebujemo, vključno z OpenAI knjižnico, dotenv knjižnico, modulom base64 in knjižnico Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Nato ustvarimo odjemalca, ki prebere API ključ iz vašega ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Naslednje generiramo sliko:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modeli vrnejo sliko kot **base64** niz v `data[0].b64_json`. Dekodiramo jo v bajte in zapišemo v datoteko — prenosa preko URL-ja ni.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Na koncu odprete sliko in jo prikažete s standardnim pregledovalnikom slik:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Več podrobnosti o generiranju slike

Oglejmo si parametre `images.generate`:

- **model**, je model slike, ki ga uporabimo (na primer `gpt-image-2`).
- **prompt**, je besedilni poziv, uporabljen za generiranje slike. Tukaj je "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, je velikost generirane slike (na primer `1024x1024`, `1536x1024`, `1024x1536` ali `"auto"`).
- **n**, je število generiranih slik. Tukaj generiramo eno.

> Modeli slik ne upoštevajo parametra `temperature` — to je kontrola za generiranje besedila. Za več različic pošljite zahtevo API znova; za manj različic naredite vaš poziv bolj specifičen.

## Dodatne možnosti generiranja slik

Videli ste, kako generirati sliko z nekaj vrsticami Pythona. `gpt-image` modeli lahko tudi **urejajo** obstoječo sliko. Z zagotovitvijo slike, opcijskega **maskiranja** (ki označuje območje za spremembo) in poziva, lahko spremenite del slike — na primer, dodate klobuk našemu zajcu.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Izvorna slika vsebuje samo zajca; končna slika ima klobuk na zajcu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
